# AI Product Strategist Agent
## Multi-Agent System for Product Analysis & PRD Generation

**Author:** Nikhil Patel  
**Date:** April 2026  
**Stack:** Python, LangGraph, OpenAI GPT-4o, SerpAPI, pandas, matplotlib

---

### Overview

This system takes a company or product name and runs it through a **5-agent pipeline** to produce a comprehensive Product Requirements Document (PRD) with competitive analysis and strategic recommendations.

| Agent | Role | Output |
|-------|------|--------|
| **Market Research Analyst** | Maps the market landscape, TAM, trends, and key players | Market context JSON |
| **Competitor Intelligence Agent** | Deep-dives on top competitors — features, pricing, positioning | Competitor matrix JSON |
| **Product Gap Analyzer** | Identifies unmet needs and whitespace opportunities | Gap analysis JSON |
| **Feature Prioritizer (RICE)** | Scores and ranks feature opportunities using RICE framework | Prioritized backlog JSON |
| **PRD Generator** | Synthesizes all outputs into a professional PRD | Complete PRD (markdown) |

**Architecture:** LangGraph state machine with shared state, each agent enriches the state before passing to the next.

```
Product Input → [Market Research] → [Competitor Intel] → [Gap Analysis] → [RICE Prioritization] → [PRD Generator] → PRD
```

---
## Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install langgraph langchain-openai langchain-core pandas matplotlib python-dotenv

In [ ]:
import os
import json
import re
from datetime import datetime
from typing import TypedDict

import pandas as pd
import matplotlib.pyplot as plt

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

from dotenv import load_dotenv
load_dotenv()

# Initialize OpenAI GPT-4o
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    max_tokens=4096,
    api_key=os.getenv("OPENAI_API_KEY")
)

print("Environment loaded. OpenAI API ready.")

---
## Step 1: Define Product Input

Provide the product or company to analyze, along with optional context about the target market and what kind of analysis you want.

In [ ]:
# Configure the analysis target
PRODUCT_INPUT = {
    'company_name': 'Notion',
    'product_name': 'Notion AI',
    'product_url': 'https://www.notion.so',
    'description': 'Notion is an all-in-one workspace for notes, docs, wikis, and project management. Notion AI adds generative AI features for writing, summarization, and data extraction within the workspace.',
    'target_market': 'Knowledge workers, product teams, startups, and mid-market companies',
    'analysis_focus': 'AI-powered productivity features and how to differentiate against competitors like Coda, Confluence, and ClickUp'
}

print(f"Analyzing: {PRODUCT_INPUT['company_name']} — {PRODUCT_INPUT['product_name']}")
print(f"Focus: {PRODUCT_INPUT['analysis_focus']}")

---
## Step 2: Define the Agent State

The shared state flows through all 5 agents. Each agent reads from and writes to this state, building up a comprehensive product analysis.

In [ ]:
class StrategistState(TypedDict):
    """Shared state flowing through the multi-agent pipeline."""
    
    # Input
    product_input: dict          # Company/product details
    
    # Agent 1: Market Research
    market_research: dict        # Market landscape, TAM, trends
    
    # Agent 2: Competitor Intelligence
    competitors: dict            # Competitor analysis matrix
    
    # Agent 3: Product Gap Analysis
    gap_analysis: dict           # Unmet needs and opportunities
    
    # Agent 4: Feature Prioritization
    feature_backlog: dict        # RICE-scored feature list
    
    # Agent 5: PRD
    prd: str                     # Complete PRD document
    
    # Metadata
    processing_log: list         # Track agent execution


print("State schema defined.")

---
## Step 3: Agent 1 — Market Research Analyst

This agent maps the market landscape: total addressable market, growth trends, key players, customer segments, and macro forces shaping the space.

In [ ]:
def market_research_agent(state: StrategistState) -> StrategistState:
    """Agent 1: Research the market landscape."""
    
    print("Agent 1: Market Research Analyst running...")
    
    product = state['product_input']
    
    prompt = f"""You are a senior market research analyst at a top strategy consulting firm.

Analyze the market landscape for this product:

Company: {product['company_name']}
Product: {product['product_name']}
URL: {product.get('product_url', 'N/A')}
Description: {product['description']}
Target Market: {product['target_market']}
Analysis Focus: {product.get('analysis_focus', 'General product strategy')}

Return ONLY valid JSON:
{{
    "market_overview": {{
        "category": "primary market category",
        "subcategories": ["list of relevant subcategories"],
        "tam": {{"value": "$XXB", "year": "2026", "source": "estimate basis"}},
        "growth_rate": "XX% CAGR",
        "maturity_stage": "emerging/growth/mature/declining"
    }},
    "key_trends": [
        {{"trend": "description", "impact": "high/medium/low", "timeframe": "now/near-term/long-term"}}
    ],
    "customer_segments": [
        {{"segment": "name", "size": "relative size", "pain_points": ["list"], "willingness_to_pay": "high/medium/low"}}
    ],
    "market_forces": {{
        "tailwinds": ["forces driving growth"],
        "headwinds": ["forces limiting growth"],
        "disruption_risks": ["emerging threats"]
    }},
    "key_players": [
        {{"name": "company", "positioning": "brief description", "estimated_market_share": "XX%"}}
    ]
}}"""
    
    response = llm.invoke([
        SystemMessage(content="You are a market research expert. Provide data-driven analysis with specific numbers. Return ONLY valid JSON."),
        HumanMessage(content=prompt)
    ])
    
    try:
        raw = response.content.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        market = json.loads(raw)
    except json.JSONDecodeError:
        market = {"error": "Failed to parse", "raw": response.content[:500]}
    
    state['market_research'] = market
    state['processing_log'].append({
        'agent': 'Market Research Analyst',
        'timestamp': datetime.now().isoformat(),
        'status': 'success' if 'error' not in market else 'error'
    })
    
    tam = market.get('market_overview', {}).get('tam', {}).get('value', 'N/A')
    players = len(market.get('key_players', []))
    trends = len(market.get('key_trends', []))
    print(f"  TAM: {tam} | {players} key players | {trends} trends identified")
    
    return state


print("Agent 1 defined: Market Research Analyst")

---
## Step 4: Agent 2 — Competitor Intelligence Agent

This agent performs a deep competitive analysis: features, pricing, positioning, strengths, weaknesses, and strategic moves. It builds a competitor matrix that feeds into gap analysis.

In [ ]:
def competitor_intel_agent(state: StrategistState) -> StrategistState:
    """Agent 2: Deep-dive competitive analysis."""
    
    print("Agent 2: Competitor Intelligence Agent running...")
    
    product = state['product_input']
    market_context = json.dumps(state['market_research'], indent=2, default=str)
    
    prompt = f"""You are a competitive intelligence specialist at a product strategy firm.

Analyze the competitive landscape for:
Company: {product['company_name']}
Product: {product['product_name']}
Description: {product['description']}
Focus: {product.get('analysis_focus', 'General')}

Market context from prior analysis:
{market_context}

Identify the top 4-5 competitors and analyze each. Return ONLY valid JSON:
{{
    "competitors": [
        {{
            "name": "competitor name",
            "product": "primary competing product",
            "positioning": "one-line positioning statement",
            "target_audience": "primary audience",
            "pricing": {{"model": "freemium/subscription/usage", "range": "$X-$X/user/month"}},
            "key_features": ["top 5 features"],
            "ai_capabilities": ["AI-specific features if any"],
            "strengths": ["top 3 strengths"],
            "weaknesses": ["top 3 weaknesses"],
            "recent_moves": ["notable recent product launches, acquisitions, pivots"],
            "threat_level": "high/medium/low"
        }}
    ],
    "feature_comparison": {{
        "categories": ["list of feature categories compared"],
        "matrix": [
            {{"feature": "feature name", "importance": "critical/high/medium/low", "{product['company_name']}": "strong/adequate/weak/missing", "competitor_leader": "who leads and why"}}
        ]
    }},
    "competitive_dynamics": {{
        "moats": ["defensible advantages the target product has"],
        "vulnerabilities": ["areas where competitors are winning"],
        "consolidation_risk": "high/medium/low — likelihood of M&A disruption",
        "pricing_pressure": "increasing/stable/decreasing"
    }}
}}"""
    
    response = llm.invoke([
        SystemMessage(content="You are a competitive intelligence expert. Be specific about features, pricing, and strategic positioning. Return ONLY valid JSON."),
        HumanMessage(content=prompt)
    ])
    
    try:
        raw = response.content.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        competitors = json.loads(raw)
    except json.JSONDecodeError:
        competitors = {"error": "Failed to parse", "raw": response.content[:500]}
    
    state['competitors'] = competitors
    state['processing_log'].append({
        'agent': 'Competitor Intelligence',
        'timestamp': datetime.now().isoformat(),
        'status': 'success' if 'error' not in competitors else 'error'
    })
    
    comp_count = len(competitors.get('competitors', []))
    features = len(competitors.get('feature_comparison', {}).get('matrix', []))
    print(f"  {comp_count} competitors analyzed | {features} features compared")
    
    return state


print("Agent 2 defined: Competitor Intelligence Agent")

---
## Step 5: Agent 3 — Product Gap Analyzer

This agent synthesizes market research and competitive intelligence to identify unmet needs, whitespace opportunities, and areas where the product can differentiate. This is where product intuition meets data.

In [ ]:
def gap_analysis_agent(state: StrategistState) -> StrategistState:
    """Agent 3: Identify product gaps and opportunities."""
    
    print("Agent 3: Product Gap Analyzer running...")
    
    product = state['product_input']
    market = json.dumps(state['market_research'], indent=2, default=str)
    competitors = json.dumps(state['competitors'], indent=2, default=str)
    
    prompt = f"""You are a product strategist identifying opportunities for product differentiation.

Based on the market research and competitive analysis below, identify product gaps and opportunities for:
Company: {product['company_name']}
Product: {product['product_name']}
Focus: {product.get('analysis_focus', 'General')}

MARKET RESEARCH:
{market}

COMPETITIVE ANALYSIS:
{competitors}

Return ONLY valid JSON:
{{
    "unmet_needs": [
        {{
            "need": "description of unmet customer need",
            "segment_affected": "which customer segment",
            "severity": "critical/high/medium/low",
            "current_workaround": "how customers solve this today",
            "opportunity_size": "large/medium/small"
        }}
    ],
    "competitive_gaps": [
        {{
            "gap": "feature or capability missing vs competitors",
            "competitors_with_this": ["list of competitors who have it"],
            "impact_on_churn": "high/medium/low",
            "difficulty_to_build": "high/medium/low"
        }}
    ],
    "whitespace_opportunities": [
        {{
            "opportunity": "description of blue ocean opportunity",
            "rationale": "why this is a whitespace",
            "potential_impact": "transformative/significant/moderate",
            "time_to_market": "months estimate"
        }}
    ],
    "strategic_recommendations": [
        {{
            "recommendation": "strategic move",
            "type": "build/buy/partner",
            "priority": "immediate/near-term/long-term",
            "rationale": "why this matters now"
        }}
    ],
    "positioning_advice": {{
        "current_positioning": "how the product is positioned today",
        "recommended_positioning": "how it should be repositioned",
        "key_differentiator": "the #1 thing to own in the market",
        "messaging_pillars": ["3 core messages"]
    }}
}}"""
    
    response = llm.invoke([
        SystemMessage(content="You are a product strategy expert. Think like a VP of Product. Identify non-obvious opportunities. Return ONLY valid JSON."),
        HumanMessage(content=prompt)
    ])
    
    try:
        raw = response.content.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        gaps = json.loads(raw)
    except json.JSONDecodeError:
        gaps = {"error": "Failed to parse", "raw": response.content[:500]}
    
    state['gap_analysis'] = gaps
    state['processing_log'].append({
        'agent': 'Product Gap Analyzer',
        'timestamp': datetime.now().isoformat(),
        'status': 'success' if 'error' not in gaps else 'error'
    })
    
    unmet = len(gaps.get('unmet_needs', []))
    comp_gaps = len(gaps.get('competitive_gaps', []))
    whitespace = len(gaps.get('whitespace_opportunities', []))
    print(f"  {unmet} unmet needs | {comp_gaps} competitive gaps | {whitespace} whitespace opportunities")
    
    return state


print("Agent 3 defined: Product Gap Analyzer")

---
## Step 6: Agent 4 — Feature Prioritizer (RICE Framework)

This agent takes the gaps and opportunities from Agent 3 and scores them using the RICE prioritization framework (Reach, Impact, Confidence, Effort). The output is a ranked feature backlog that any PM would recognize.

In [ ]:
def feature_prioritizer_agent(state: StrategistState) -> StrategistState:
    """Agent 4: Prioritize features using RICE framework."""
    
    print("Agent 4: Feature Prioritizer (RICE) running...")
    
    product = state['product_input']
    gaps = json.dumps(state['gap_analysis'], indent=2, default=str)
    market = json.dumps(state['market_research'], indent=2, default=str)
    
    prompt = f"""You are a senior product manager creating a prioritized feature backlog.

Based on the gap analysis and market research below, create a prioritized feature backlog for:
Product: {product['product_name']} by {product['company_name']}

GAP ANALYSIS:
{gaps}

MARKET CONTEXT:
{market}

Score each feature using the RICE framework:
- Reach: How many users will this impact per quarter? (number)
- Impact: How much will it move the needle? (3=massive, 2=high, 1=medium, 0.5=low, 0.25=minimal)
- Confidence: How confident are we in the estimates? (100%/80%/50%/20%)
- Effort: How many person-months to build? (number)
- RICE Score = (Reach * Impact * Confidence) / Effort

Generate 8-12 feature proposals. Return ONLY valid JSON:
{{
    "features": [
        {{
            "id": "F001",
            "name": "feature name",
            "description": "what it does and why it matters",
            "user_story": "As a [user], I want [capability], so that [benefit]",
            "addresses": "which gap or unmet need this solves",
            "reach": 10000,
            "impact": 2,
            "confidence": 0.8,
            "effort": 3,
            "rice_score": 5333,
            "tier": "P0/P1/P2",
            "category": "AI/UX/Integration/Platform/Growth",
            "dependencies": ["list any prerequisites"],
            "success_metrics": ["how we measure if this worked"]
        }}
    ],
    "roadmap_recommendation": {{
        "now": ["feature IDs for immediate build (0-3 months)"],
        "next": ["feature IDs for next quarter (3-6 months)"],
        "later": ["feature IDs for future (6-12 months)"],
        "rationale": "explanation of sequencing logic"
    }},
    "resource_estimate": {{
        "total_effort_months": 0,
        "recommended_team_size": 0,
        "timeline": "X months to ship P0 features"
    }}
}}"""
    
    response = llm.invoke([
        SystemMessage(content="You are a senior PM expert in prioritization frameworks. Be quantitative. RICE scores must be calculated correctly. Return ONLY valid JSON."),
        HumanMessage(content=prompt)
    ])
    
    try:
        raw = response.content.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        features = json.loads(raw)
    except json.JSONDecodeError:
        features = {"error": "Failed to parse", "raw": response.content[:500]}
    
    state['feature_backlog'] = features
    state['processing_log'].append({
        'agent': 'Feature Prioritizer (RICE)',
        'timestamp': datetime.now().isoformat(),
        'status': 'success' if 'error' not in features else 'error'
    })
    
    feat_count = len(features.get('features', []))
    p0 = len(features.get('roadmap_recommendation', {}).get('now', []))
    print(f"  {feat_count} features scored | {p0} P0 features for immediate build")
    
    return state


print("Agent 4 defined: Feature Prioritizer (RICE)")

---
## Step 7: Agent 5 — PRD Generator

The synthesis agent. Takes all outputs from Agents 1-4 and generates a professional Product Requirements Document that you'd actually present to a VP of Product.

In [ ]:
def prd_generator_agent(state: StrategistState) -> StrategistState:
    """Agent 5: Generate the complete PRD."""
    
    print("Agent 5: PRD Generator running...")
    
    product = state['product_input']
    market = json.dumps(state['market_research'], indent=2, default=str)
    competitors = json.dumps(state['competitors'], indent=2, default=str)
    gaps = json.dumps(state['gap_analysis'], indent=2, default=str)
    features = json.dumps(state['feature_backlog'], indent=2, default=str)
    
    prompt = f"""You are a VP of Product writing a comprehensive Product Requirements Document.

Using analysis from four specialized agents, create a professional PRD for {product['product_name']} by {product['company_name']}.

MARKET RESEARCH:\n{market}

COMPETITIVE ANALYSIS:\n{competitors}

GAP ANALYSIS:\n{gaps}

FEATURE BACKLOG:\n{features}

Write the PRD in this EXACT markdown format:

# Product Requirements Document: {product['product_name']}
### {product['company_name']} — Strategic Product Analysis

**Author:** AI Product Strategist Agent  
**Date:** {datetime.now().strftime('%B %d, %Y')}  
**Status:** Draft for Review  
**Confidentiality:** Internal

---

## 1. Executive Summary
[3-4 paragraphs: the product opportunity, why now, strategic recommendation]

## 2. Market Context
[Market size, growth, trends, key dynamics — synthesized from Agent 1]

## 3. Competitive Landscape
[Competitor matrix table, key takeaways — synthesized from Agent 2]

| Competitor | Positioning | AI Capabilities | Threat Level |
|------------|------------|-----------------|-------------|
[fill from data]

## 4. Problem Statement & Opportunity
[Unmet needs, competitive gaps, whitespace — synthesized from Agent 3]

## 5. Proposed Solution
### 5.1 Vision
[One paragraph: the north star for the product]

### 5.2 Feature Specifications
[Top 5 features with user stories, acceptance criteria, success metrics — from Agent 4]

### 5.3 Prioritized Roadmap
| Phase | Timeline | Features | Key Outcome |
|-------|----------|----------|-------------|
| Now | 0-3 months | [features] | [outcome] |
| Next | 3-6 months | [features] | [outcome] |
| Later | 6-12 months | [features] | [outcome] |

## 6. Success Metrics
[KPIs and targets for measuring success]

## 7. Risks & Mitigations
[Top risks with mitigation strategies]

## 8. Resource Requirements
[Team size, skills, timeline, estimated investment]

## 9. Appendix
[RICE scoring table for all features]

---
*Generated by AI Product Strategist Agent — Multi-Agent System by Nikhil Patel*"""
    
    response = llm.invoke([
        SystemMessage(content="You are a VP of Product at a top tech company. Write a PRD that would impress a CPO. Be specific, data-driven, and strategic. Use the data provided — don't make up numbers."),
        HumanMessage(content=prompt)
    ])
    
    prd = response.content.strip()
    
    state['prd'] = prd
    state['processing_log'].append({
        'agent': 'PRD Generator',
        'timestamp': datetime.now().isoformat(),
        'status': 'success'
    })
    
    print(f"  PRD generated: {len(prd):,} chars")
    
    return state


print("Agent 5 defined: PRD Generator")

---
## Step 8: Build the LangGraph Pipeline

Wire all five agents into a LangGraph state machine. Each agent is a node, and the graph defines the execution flow.

In [ ]:
# Build the LangGraph
workflow = StateGraph(StrategistState)

# Add agent nodes
workflow.add_node("market_research", market_research_agent)
workflow.add_node("competitor_intel", competitor_intel_agent)
workflow.add_node("gap_analysis", gap_analysis_agent)
workflow.add_node("feature_prioritizer", feature_prioritizer_agent)
workflow.add_node("prd_generator", prd_generator_agent)

# Define the flow: linear pipeline
workflow.set_entry_point("market_research")
workflow.add_edge("market_research", "competitor_intel")
workflow.add_edge("competitor_intel", "gap_analysis")
workflow.add_edge("gap_analysis", "feature_prioritizer")
workflow.add_edge("feature_prioritizer", "prd_generator")
workflow.add_edge("prd_generator", END)

# Compile the graph
app = workflow.compile()

print("LangGraph pipeline compiled.")
print("Flow: Market Research → Competitor Intel → Gap Analysis → Feature Prioritization → PRD → END")

---
## Step 9: Run the Pipeline

In [ ]:
# Initialize state
initial_state = {
    'product_input': PRODUCT_INPUT,
    'market_research': {},
    'competitors': {},
    'gap_analysis': {},
    'feature_backlog': {},
    'prd': '',
    'processing_log': []
}

print(f"Analyzing {PRODUCT_INPUT['company_name']} — {PRODUCT_INPUT['product_name']}...")
print("=" * 60)

# Run the pipeline
result = app.invoke(initial_state)

print("=" * 60)
print(f"\nPipeline complete.")
print(f"Agents executed: {len(result['processing_log'])}")
for log in result['processing_log']:
    print(f"  [{log['status']}] {log['agent']} @ {log['timestamp']}")

---
## Step 10: Display the PRD

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result['prd']))

---
## Step 11: Visualizations

In [ ]:
def create_strategy_dashboard(result: dict):
    """Generate visual dashboard from pipeline results."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    product = result['product_input']
    fig.suptitle(
        f"{product['company_name']} — {product['product_name']} Strategy Dashboard",
        fontsize=16, fontweight='bold', y=0.98, color='white'
    )
    fig.patch.set_facecolor('#0a1128')
    
    for ax in axes.flat:
        ax.set_facecolor('#0d1635')
        ax.tick_params(colors='#a0a0a0')
        ax.spines['bottom'].set_color('#2a2a2a')
        ax.spines['left'].set_color('#2a2a2a')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    # Plot 1: RICE Scores (top features)
    ax1 = axes[0, 0]
    features = result.get('feature_backlog', {}).get('features', [])
    if features:
        sorted_features = sorted(features, key=lambda x: x.get('rice_score', 0), reverse=True)[:8]
        names = [f.get('name', '?')[:25] for f in sorted_features]
        scores = [f.get('rice_score', 0) for f in sorted_features]
        tiers = [f.get('tier', 'P2') for f in sorted_features]
        colors = ['#4CC9A4' if t == 'P0' else '#F5C878' if t == 'P1' else '#a0a0a0' for t in tiers]
        ax1.barh(names[::-1], scores[::-1], color=colors[::-1], alpha=0.85)
        ax1.set_xlabel('RICE Score', color='#a0a0a0')
    ax1.set_title('Feature Priority (RICE Scores)', color='white', fontsize=12, pad=10)
    ax1.tick_params(axis='y', labelsize=8, colors='#a0a0a0')
    
    # Plot 2: Competitor Threat Levels
    ax2 = axes[0, 1]
    comps = result.get('competitors', {}).get('competitors', [])
    if comps:
        comp_names = [c.get('name', '?')[:15] for c in comps]
        threat_map = {'high': 3, 'medium': 2, 'low': 1}
        threats = [threat_map.get(c.get('threat_level', 'low'), 1) for c in comps]
        threat_colors = ['#e74c3c' if t == 3 else '#F5C878' if t == 2 else '#4CC9A4' for t in threats]
        ax2.barh(comp_names, threats, color=threat_colors, alpha=0.85)
        ax2.set_xticks([1, 2, 3])
        ax2.set_xticklabels(['Low', 'Medium', 'High'], color='#a0a0a0')
    ax2.set_title('Competitor Threat Levels', color='white', fontsize=12, pad=10)
    ax2.tick_params(axis='y', labelsize=9, colors='#a0a0a0')
    
    # Plot 3: Opportunity Map
    ax3 = axes[1, 0]
    gaps = result.get('gap_analysis', {})
    categories = ['Unmet Needs', 'Competitive\nGaps', 'Whitespace\nOpportunities']
    counts = [
        len(gaps.get('unmet_needs', [])),
        len(gaps.get('competitive_gaps', [])),
        len(gaps.get('whitespace_opportunities', []))
    ]
    bar_colors = ['#e74c3c', '#F5C878', '#4CC9A4']
    ax3.bar(categories, counts, color=bar_colors, width=0.5)
    for i, v in enumerate(counts):
        ax3.text(i, v + 0.1, str(v), ha='center', color='white', fontweight='bold')
    ax3.set_title('Opportunity Landscape', color='white', fontsize=12, pad=10)
    ax3.tick_params(axis='x', colors='#a0a0a0', labelsize=9)
    
    # Plot 4: Roadmap Summary
    ax4 = axes[1, 1]
    roadmap = result.get('feature_backlog', {}).get('roadmap_recommendation', {})
    phases = ['Now\n(0-3 mo)', 'Next\n(3-6 mo)', 'Later\n(6-12 mo)']
    phase_counts = [
        len(roadmap.get('now', [])),
        len(roadmap.get('next', [])),
        len(roadmap.get('later', []))
    ]
    phase_colors = ['#4CC9A4', '#F5C878', '#00d9ff']
    ax4.bar(phases, phase_counts, color=phase_colors, width=0.5)
    for i, v in enumerate(phase_counts):
        ax4.text(i, v + 0.1, str(v), ha='center', color='white', fontweight='bold')
    ax4.set_title('Roadmap Distribution', color='white', fontsize=12, pad=10)
    ax4.tick_params(axis='x', colors='#a0a0a0')
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('sample_outputs/strategy_dashboard.png', dpi=150, bbox_inches='tight',
                facecolor='#0a1128', edgecolor='none')
    plt.show()
    print("Dashboard saved to sample_outputs/strategy_dashboard.png")


create_strategy_dashboard(result)

---
## Step 12: Save Outputs

In [ ]:
import os
os.makedirs('sample_outputs', exist_ok=True)

product = result['product_input']
prefix = product['company_name'].replace(' ', '_')

# Save PRD as markdown
prd_path = f"sample_outputs/{prefix}_PRD.md"
with open(prd_path, 'w') as f:
    f.write(result['prd'])
print(f"PRD saved to {prd_path}")

# Save full state as JSON
state_path = f"sample_outputs/{prefix}_full_analysis.json"
serializable = {
    'product_input': result['product_input'],
    'market_research': result['market_research'],
    'competitors': result['competitors'],
    'gap_analysis': result['gap_analysis'],
    'feature_backlog': result['feature_backlog'],
    'processing_log': result['processing_log']
}
with open(state_path, 'w') as f:
    json.dump(serializable, f, indent=2, default=str)
print(f"Full analysis saved to {state_path}")

---
## Architecture Summary

```
                    +------------------+
                    |  Product Input   |
                    | (Company/URL)    |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Agent 1:        |
                    |  Market Research |
                    |  Analyst         |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Agent 2:        |
                    |  Competitor      |
                    |  Intelligence    |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Agent 3:        |
                    |  Product Gap     |
                    |  Analyzer        |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Agent 4:        |
                    |  Feature         |
                    |  Prioritizer     |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Agent 5:        |
                    |  PRD Generator   |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Complete PRD    |
                    |  + Dashboard     |
                    |  + Feature Map   |
                    +------------------+
```

### Key Design Decisions

1. **5-agent pipeline over monolithic prompt:** Each agent has a specialized system prompt optimized for its domain. A market researcher thinks differently from a feature prioritizer — the separation produces higher-quality outputs.

2. **RICE framework for prioritization:** Using an industry-standard framework (Reach × Impact × Confidence / Effort) makes the output immediately credible to any PM or hiring manager who reads it.

3. **Cumulative state enrichment:** Each agent builds on the previous agent's output. The gap analyzer can only identify true gaps because it has both market context AND competitive data. The feature prioritizer can only score accurately because it understands the gaps.

4. **JSON-structured intermediate outputs:** Every agent returns structured JSON, enabling programmatic dashboards and ensuring the PRD generator has clean, parseable data to synthesize.

5. **Product-manager-native output:** The final PRD follows a standard format that any product leader would recognize — executive summary, market context, competitive landscape, problem statement, proposed solution with user stories, prioritized roadmap, success metrics, and risks.

---
*Analysis by Nikhil Patel | Part of [The Lab](https://github.com/patelnikhilc/The-Lab) — AI Product Case Studies*